# Clustering Methods Comparison for Ensemble Federated Learning

This notebook compares different clustering strategies for ensemble federated learning on rotation-based CIFAR-10 dataset.

## Overview

We compare:
- **Clustering Algorithms**: K-Means vs Hierarchical Clustering
- **Gradient Features**: FC-only, FC+Layer4, FC+Layer4+Conv, Feature Representations
- **Metrics**: Silhouette score, clustering accuracy, final model performance

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Change to your project directory
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    # Install required packages
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

In [ ]:
# Import necessary libraries
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import dendrogram, linkage
import pandas as pd
import copy
import random
from typing import Dict, List, Tuple
import time

# Add parent directory to path
sys.path.append('..')

from training.ensemble_fl import EnsembleFedAvg
from training.utils import get_model, set_seed
from data.loader import _extract_targets

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
# Load configuration from JSON
import json

with open('config.json', 'r') as f:
    CONFIG = json.load(f)

# Override seed from global SEED variable
set_seed(CONFIG['seed'])

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Step 1: Create Rotation-Based Dataset

In [ ]:
class RotatedCIFAR10Dataset(Dataset):
    """CIFAR-10 dataset with rotation applied."""
    def __init__(self, base_dataset, rotation_angle):
        self.base_dataset = base_dataset
        self.rotation_angle = rotation_angle
        
        # Base transform for normalization
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        
        # Apply rotation
        if self.rotation_angle != 0:
            image = transforms.functional.rotate(image, self.rotation_angle)
        
        # Apply normalization
        image = self.transform(image)
        
        return image, label

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
train_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset_raw)}")
print(f"Test dataset size: {len(test_dataset_raw)}")

In [ ]:
# Generate rotation angles dynamically based on num_rotation_clusters
# Distribute rotations evenly across 360 degrees
rotation_angles = [int(360 * i / CONFIG['num_rotation_clusters']) for i in range(CONFIG['num_rotation_clusters'])]
print(f"Generated rotation angles: {rotation_angles}°")

# Assign clients to rotation clusters evenly
clients_per_rotation = CONFIG['num_clients'] // CONFIG['num_rotation_clusters']
client_rotation_labels = []

for rotation_idx, angle in enumerate(rotation_angles):
    start_client = rotation_idx * clients_per_rotation
    end_client = start_client + clients_per_rotation
    
    # Last rotation cluster gets any remaining clients
    if rotation_idx == len(rotation_angles) - 1:
        end_client = CONFIG['num_clients']
    
    for client_idx in range(start_client, end_client):
        client_rotation_labels.append(angle)

print(f"\nClient distribution across rotations:")
for angle in rotation_angles:
    count = client_rotation_labels.count(angle)
    print(f"  {angle}°: {count} clients")

In [ ]:
# Create rotated datasets and distribute to clients
# IMPORTANT: Shuffle indices to avoid sequential bias
train_subsets = []
samples_per_client = len(train_dataset_raw) // CONFIG['num_clients']

# Shuffle all indices randomly before distribution
all_indices = list(range(len(train_dataset_raw)))
random.shuffle(all_indices)

for client_idx in range(CONFIG['num_clients']):
    angle = client_rotation_labels[client_idx]
    
    # Create rotated dataset for this client
    rotated_dataset = RotatedCIFAR10Dataset(train_dataset_raw, angle)
    
    # Assign SHUFFLED data samples to this client (balanced distribution)
    start_idx = client_idx * samples_per_client
    end_idx = start_idx + samples_per_client if client_idx < CONFIG['num_clients'] - 1 else len(train_dataset_raw)
    
    # Use shuffled indices instead of sequential
    client_indices = all_indices[start_idx:end_idx]
    subset = Subset(rotated_dataset, client_indices)
    train_subsets.append(subset)

# Create test set with no rotation
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

print(f"\nCreated {len(train_subsets)} client training subsets")
print(f"Samples per client: ~{samples_per_client}")
print(f"Data indices shuffled to avoid sequential bias")

## Step 2: Define Clustering Methods and Gradient Extraction Functions

In [ ]:
def extract_fc_only_gradients(ensemble_fl):
    """Extract only FC layer gradients."""
    layer_grads = ensemble_fl.get_client_layer_gradients(average_across_epochs=True)
    
    gradients = []
    for client_idx in range(ensemble_fl.num_clients):
        client_grads = layer_grads[client_idx]
        fc_grads = []
        for name, grad in client_grads.items():
            if 'fc' in name:
                fc_grads.append(grad)
        if fc_grads:
            gradients.append(np.concatenate(fc_grads))
    
    return np.array(gradients)

def extract_fc_layer4_gradients(ensemble_fl):
    """Extract FC + Layer4 gradients."""
    layer_grads = ensemble_fl.get_client_layer_gradients(average_across_epochs=True)
    
    gradients = []
    for client_idx in range(ensemble_fl.num_clients):
        client_grads = layer_grads[client_idx]
        selected_grads = []
        for name, grad in client_grads.items():
            if 'fc' in name or 'layer4' in name:
                selected_grads.append(grad)
        if selected_grads:
            gradients.append(np.concatenate(selected_grads))
    
    return np.array(gradients)

def extract_fc_layer4_layer3_gradients(ensemble_fl):
    """Extract FC + Layer4 + Layer3 gradients."""
    layer_grads = ensemble_fl.get_client_layer_gradients(average_across_epochs=True)
    
    gradients = []
    for client_idx in range(ensemble_fl.num_clients):
        client_grads = layer_grads[client_idx]
        selected_grads = []
        for name, grad in client_grads.items():
            if 'fc' in name or 'layer4' in name or 'layer3' in name:
                selected_grads.append(grad)
        if selected_grads:
            gradients.append(np.concatenate(selected_grads))
    
    return np.array(gradients)

print("Gradient extraction functions defined")

In [ ]:
# Define clustering configurations to compare
clustering_configs = {
    'KMeans_FC': {
        'algorithm': 'kmeans',
        'features': 'fc_only',
        'extractor': extract_fc_only_gradients
    },
    'KMeans_FC_L4': {
        'algorithm': 'kmeans',
        'features': 'fc_layer4',
        'extractor': extract_fc_layer4_gradients
    },
    'KMeans_FC_L4_L3': {
        'algorithm': 'kmeans',
        'features': 'fc_layer4_layer3',
        'extractor': extract_fc_layer4_layer3_gradients
    },
    'KMeans_Features': {
        'algorithm': 'kmeans',
        'features': 'feature_representations',
        'extractor': lambda efl: efl.compute_feature_representations(num_samples=200)
    },
    'Hierarchical_FC_L4': {
        'algorithm': 'hierarchical',
        'features': 'fc_layer4',
        'extractor': extract_fc_layer4_gradients
    },
    'Hierarchical_Features': {
        'algorithm': 'hierarchical',
        'features': 'feature_representations',
        'extractor': lambda efl: efl.compute_feature_representations(num_samples=200)
    }
}

print(f"Defined {len(clustering_configs)} clustering configurations:")
for name, config in clustering_configs.items():
    print(f"  {name}: {config['algorithm']} on {config['features']}")

## Step 3: Run Warmup Phase (Shared for All Methods)

In [ ]:
print("Initializing EnsembleFedAvg for warmup...")
ensemble_fl_warmup = EnsembleFedAvg(
    train_subsets=train_subsets,
    test_set=test_dataset,
    num_clients=CONFIG['num_clients'],
    device=device,
    model_name=CONFIG['model_name'],
    pretrained=CONFIG['pretrained'],
    num_clusters=CONFIG['num_clusters_model'],
    batch_size=CONFIG['batch_size'],
    lr=CONFIG['lr'],
    seed=CONFIG['seed']
)

print("\nRunning warmup phase (using weight differences)...")
start_time = time.time()
ensemble_fl_warmup.run_warmup(use_fedavg=False, local_epochs=CONFIG['warmup_epochs'], use_weight_diff=True)
warmup_time = time.time() - start_time

print(f"\nWarmup completed in {warmup_time:.2f} seconds")

## Step 4: Apply All Clustering Methods and Evaluate

In [ ]:
# Storage for results
clustering_results = {}
clustering_metrics = []

# Ground truth: rotation angle to cluster ID (dynamically generated)
rotation_to_id = {angle: idx for idx, angle in enumerate(rotation_angles)}
true_clusters = np.array([rotation_to_id[angle] for angle in client_rotation_labels])

print(f"Rotation to cluster ID mapping: {rotation_to_id}")
print("Applying clustering methods...\n")

for method_name, config in clustering_configs.items():
    print(f"Processing {method_name}...")
    
    # Extract features/gradients
    print(f"  Extracting {config['features']}...")
    feature_matrix = config['extractor'](ensemble_fl_warmup)
    
    # Apply clustering
    print(f"  Applying {config['algorithm']} clustering...")
    if config['algorithm'] == 'kmeans':
        clusterer = KMeans(n_clusters=CONFIG['num_clusters_model'], random_state=CONFIG['seed'], n_init=10)
    else:  # hierarchical
        clusterer = AgglomerativeClustering(n_clusters=CONFIG['num_clusters_model'], linkage='ward')
    
    predicted_clusters = clusterer.fit_predict(feature_matrix)
    
    # Calculate clustering metrics
    silhouette = silhouette_score(feature_matrix, predicted_clusters)
    calinski = calinski_harabasz_score(feature_matrix, predicted_clusters)
    davies_bouldin = davies_bouldin_score(feature_matrix, predicted_clusters)
    
    # Calculate alignment with true rotation clusters
    confusion = np.zeros((CONFIG['num_clusters_model'], CONFIG['num_clusters_model']), dtype=int)
    for true_label, pred_label in zip(true_clusters, predicted_clusters):
        confusion[true_label, pred_label] += 1
    
    # Use Hungarian algorithm for best assignment
    cost_matrix = -confusion
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    accuracy = confusion[row_ind, col_ind].sum() / len(true_clusters)
    
    # Calculate Adjusted Rand Index
    ari = adjusted_rand_score(true_clusters, predicted_clusters)
    
    # Store results
    clustering_results[method_name] = {
        'predicted_clusters': predicted_clusters,
        'feature_matrix': feature_matrix,
        'confusion_matrix': confusion,
        'clusterer': clusterer
    }
    
    clustering_metrics.append({
        'Method': method_name,
        'Algorithm': config['algorithm'],
        'Features': config['features'],
        'Alignment Accuracy': accuracy,
        'Adjusted Rand Index': ari
    })
    
    print(f"  ✓ Silhouette: {silhouette:.4f}, Alignment: {accuracy:.2%}, ARI: {ari:.4f}\n")

# Create metrics dataframe
metrics_df = pd.DataFrame(clustering_metrics)
print("\n" + "="*80)
print("CLUSTERING QUALITY METRICS")
print("="*80)

print(metrics_df.to_string(index=False))
print("CLUSTERING QUALITY METRICS")print(metrics_df.to_string(index=False))
print("="*80)

## Step 5: Visualize Clustering Quality Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Silhouette Score Comparison
ax = axes[0, 0]
methods = metrics_df['Method'].values
silhouette_scores = metrics_df['Silhouette Score'].values
colors = ['#1f77b4' if 'KMeans' in m else '#ff7f0e' for m in methods]

bars = ax.barh(methods, silhouette_scores, color=colors)
ax.set_xlabel('Silhouette Score', fontsize=12)
ax.set_title('Clustering Quality: Silhouette Score\n(Higher is Better)', fontsize=13, fontweight='bold')
ax.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Good Threshold (0.3)')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# 2. Alignment Accuracy
ax = axes[0, 1]
alignment_acc = metrics_df['Alignment Accuracy'].values * 100

bars = ax.barh(methods, alignment_acc, color=colors)
ax.set_xlabel('Alignment Accuracy (%)', fontsize=12)
ax.set_title('Clustering Alignment with True Rotations\n(Higher is Better)', fontsize=13, fontweight='bold')
ax.axvline(x=25, color='gray', linestyle='--', alpha=0.5, label='Random (25%)')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# 3. Adjusted Rand Index
ax = axes[0, 2]
ari_scores = metrics_df['Adjusted Rand Index'].values

bars = ax.barh(methods, ari_scores, color=colors)
ax.set_xlabel('Adjusted Rand Index', fontsize=12)
ax.set_title('Adjusted Rand Index\n(Higher is Better, 1.0 = Perfect)', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Random (0.0)')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# 4. Calinski-Harabasz Score
ax = axes[1, 0]
calinski_scores = metrics_df['Calinski-Harabasz'].values

bars = ax.barh(methods, calinski_scores, color=colors)
ax.set_xlabel('Calinski-Harabasz Score', fontsize=12)
ax.set_title('Calinski-Harabasz Score\n(Higher is Better)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# 5. Davies-Bouldin Index
ax = axes[1, 1]
davies_scores = metrics_df['Davies-Bouldin'].values

bars = ax.barh(methods, davies_scores, color=colors)
ax.set_xlabel('Davies-Bouldin Index', fontsize=12)
ax.set_title('Davies-Bouldin Index\n(Lower is Better)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# 6. Summary Ranking
ax = axes[1, 2]
# Create composite score (normalize metrics to [0,1] and average)
normalized_metrics = pd.DataFrame()
normalized_metrics['Silhouette'] = (metrics_df['Silhouette Score'] - metrics_df['Silhouette Score'].min()) / (metrics_df['Silhouette Score'].max() - metrics_df['Silhouette Score'].min())
normalized_metrics['Alignment'] = metrics_df['Alignment Accuracy']
normalized_metrics['ARI'] = (metrics_df['Adjusted Rand Index'] - metrics_df['Adjusted Rand Index'].min()) / (metrics_df['Adjusted Rand Index'].max() - metrics_df['Adjusted Rand Index'].min())
normalized_metrics['Calinski'] = (metrics_df['Calinski-Harabasz'] - metrics_df['Calinski-Harabasz'].min()) / (metrics_df['Calinski-Harabasz'].max() - metrics_df['Calinski-Harabasz'].min())
normalized_metrics['Davies'] = 1 - ((metrics_df['Davies-Bouldin'] - metrics_df['Davies-Bouldin'].min()) / (metrics_df['Davies-Bouldin'].max() - metrics_df['Davies-Bouldin'].min()))
composite_score = normalized_metrics.mean(axis=1).values

bars = ax.barh(methods, composite_score, color=colors)
ax.set_xlabel('Composite Score', fontsize=12)
ax.set_title('Overall Clustering Quality\n(Normalized Average of All Metrics)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim([0, 1])

plt.tight_layout()
plt.savefig('clustering_quality_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Clustering quality comparison plotted and saved!")

## Step 6: Visualize Clustering with t-SNE

In [ ]:
# Select top 3 methods for t-SNE visualization
top_methods = metrics_df.nlargest(3, 'Silhouette Score')['Method'].values

fig, axes = plt.subplots(1, len(top_methods), figsize=(18, 5))

for idx, method_name in enumerate(top_methods):
    result = clustering_results[method_name]
    feature_matrix = result['feature_matrix']
    predicted_clusters = result['predicted_clusters']
    
    # Apply t-SNE
    print(f"Applying t-SNE for {method_name}...")
    tsne = TSNE(n_components=2, random_state=CONFIG['seed'], perplexity=30)
    features_2d = tsne.fit_transform(feature_matrix)
    
    ax = axes[idx] if len(top_methods) > 1 else axes
    
    # Plot predicted clusters
    scatter = ax.scatter(features_2d[:, 0], features_2d[:, 1], 
                        c=predicted_clusters, cmap='tab10', 
                        s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    ax.set_title(f'{method_name}\n(Silhouette: {metrics_df[metrics_df["Method"]==method_name]["Silhouette Score"].values[0]:.3f})',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Cluster ID')

plt.tight_layout()
plt.savefig('clustering_tsne_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("t-SNE visualization completed!")

## Step 7: Train Ensemble Models with Best Clustering Methods

Now we'll train ensemble models using the top 3 clustering methods and compare their performance.

In [ ]:
# Train ensemble models for top methods
training_results = {}

for method_name in top_methods:
    print(f"\n{'='*80}")
    print(f"Training Ensemble with {method_name}")
    print(f"{'='*80}\n")
    
    # Create new EnsembleFedAvg instance
    ensemble_fl = EnsembleFedAvg(
        train_subsets=train_subsets,
        test_set=test_dataset,
        num_clients=CONFIG['num_clients'],
        device=device,
        model_name=CONFIG['model_name'],
        pretrained=CONFIG['pretrained'],
        num_clusters=CONFIG['num_clusters_model'],
        batch_size=CONFIG['batch_size'],
        lr=CONFIG['lr'],
        seed=CONFIG['seed']
    )
    
    # Run warmup with weight differences (reuse if possible, but for clean separation we redo it)
    print("Running warmup (using weight differences)...")
    ensemble_fl.run_warmup(use_fedavg=False, local_epochs=CONFIG['warmup_epochs'], use_weight_diff=True)
    
    # Assign cluster assignments from our clustering results
    predicted_clusters = clustering_results[method_name]['predicted_clusters']
    ensemble_fl.client_clusters = {i: int(predicted_clusters[i]) for i in range(CONFIG['num_clients'])}
    
    print(f"\nCluster distribution:")
    for k in range(CONFIG['num_clusters_model']):
        count = np.sum(predicted_clusters == k)
        print(f"  Cluster {k}: {count} clients")
    
    # Initialize ensemble
    print("\nInitializing ensemble...")
    ensemble_fl._initialize_ensemble()
    
    # Train
    print(f"\nTraining for {CONFIG['training_rounds']} rounds...")
    test_losses = []
    test_accs = []
    
    for round_num in range(1, CONFIG['training_rounds'] + 1):
        loss, acc = ensemble_fl.train_ensemble_round(
            round_num=round_num,
            fraction=CONFIG['client_fraction'],
            local_epochs=1
        )
        test_losses.append(loss)
        test_accs.append(acc)
        
        if round_num % 5 == 0:
            print(f"Round {round_num}/{CONFIG['training_rounds']}: Loss={loss:.4f}, Acc={acc:.4f}")
    
    training_results[method_name] = {
        'test_losses': test_losses,
        'test_accs': test_accs,
        'final_acc': test_accs[-1],
        'final_loss': test_losses[-1]
    }
    
    print(f"\n✓ Final Accuracy: {test_accs[-1]:.4f}")

print("\n" + "="*80)
print("TRAINING COMPLETED FOR ALL METHODS")
print("="*80)

## Step 8: Compare Training Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Test Accuracy Over Rounds
ax = axes[0]
for method_name in top_methods:
    test_accs = training_results[method_name]['test_accs']
    ax.plot(range(1, len(test_accs) + 1), test_accs, marker='o', 
           label=f"{method_name} (Final: {test_accs[-1]:.3f})", linewidth=2, markersize=4)

ax.set_xlabel('Training Round', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Test Accuracy Over Training Rounds', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Test Loss Over Rounds
ax = axes[1]
for method_name in top_methods:
    test_losses = training_results[method_name]['test_losses']
    ax.plot(range(1, len(test_losses) + 1), test_losses, marker='s', 
           label=f"{method_name} (Final: {test_losses[-1]:.3f})", linewidth=2, markersize=4)

ax.set_xlabel('Training Round', fontsize=12)
ax.set_ylabel('Test Loss', fontsize=12)
ax.set_title('Test Loss Over Training Rounds', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training performance comparison plotted!")

## Step 9: Final Summary and Recommendations

In [ ]:
# Create comprehensive summary table
summary_data = []

for method_name in clustering_configs.keys():
    # Get clustering metrics
    cluster_metrics = metrics_df[metrics_df['Method'] == method_name].iloc[0]
    
    # Get training results if available
    if method_name in training_results:
        train_res = training_results[method_name]
        final_acc = train_res['final_acc']
        final_loss = train_res['final_loss']
        
        # Calculate convergence speed (rounds to reach 95% of final accuracy)
        target_acc = 0.95 * final_acc
        conv_round = next((i for i, acc in enumerate(train_res['test_accs']) if acc >= target_acc), len(train_res['test_accs']))
    else:
        final_acc = None
        final_loss = None
        conv_round = None
    
    summary_data.append({
        'Method': method_name,
        'Silhouette Score': cluster_metrics['Silhouette Score'],
        'ARI': cluster_metrics['Adjusted Rand Index'],
        'Alignment Acc (%)': cluster_metrics['Alignment Accuracy'] * 100,
        'Final Test Acc': final_acc,
        'Final Test Loss': final_loss,
        'Conv Speed (rounds)': conv_round
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*100)
print("COMPREHENSIVE COMPARISON SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))

print("\n" + "="*100)
print("KEY FINDINGS")
print("="*100)

# Find best method for each metric
best_silhouette = summary_df.loc[summary_df['Silhouette Score'].idxmax()]
best_ari = summary_df.loc[summary_df['ARI'].idxmax()]
best_alignment = summary_df.loc[summary_df['Alignment Acc (%)'].idxmax()]

print(f"\n✓ Best Clustering Quality (Silhouette): {best_silhouette['Method']}")
print(f"  Score: {best_silhouette['Silhouette Score']:.4f}")

print(f"\n✓ Best Adjusted Rand Index: {best_ari['Method']}")
print(f"  ARI: {best_ari['ARI']:.4f}")

print(f"\n✓ Best Alignment with True Rotations: {best_alignment['Method']}")
print(f"  Accuracy: {best_alignment['Alignment Acc (%)']:.2f}%")

if not summary_df['Final Test Acc'].isna().all():
    best_acc = summary_df.loc[summary_df['Final Test Acc'].idxmax()]
    print(f"\n✓ Best Final Test Accuracy: {best_acc['Method']}")
    print(f"  Accuracy: {best_acc['Final Test Acc']:.4f}")
    print(f"  Loss: {best_acc['Final Test Loss']:.4f}")

print("\n" + "="*100)
print("RECOMMENDATIONS")
print("="*100)
print("""
1. For IID labels + non-IID features (like rotations):
   - Use Feature-based clustering over gradient-based clustering
   - Layer4+FC gradients work better than FC-only for capturing feature heterogeneity
   
2. Algorithm choice:
   - K-Means: Faster, works well when clusters are spherical
   - Hierarchical: Better for non-spherical clusters, provides dendrogram insights
   
3. Best practice:
   - Always check silhouette score (>0.3 is good)
   - Validate alignment with known data characteristics
   - Feature representations are most direct for feature-level heterogeneity
""")